In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv("/kaggle/input/dataset-mba-decision-after-bachelors/mba_decision_dataset.csv")

In [3]:
df = df.loc[
    ~((df['Age'] .isin([21, 22,23,24,25])) & (df['Current Job Title'] == 'Manager'))
]
#new age
df['modified-age'] = df['Age']
df.loc[(df['Age'] == 21) & (df['Years of Work Experience'] > 1), 'modified-age'] = df['Age'] + df['Years of Work Experience']
df.loc[(df['Age'] == 22) & (df['Years of Work Experience'] > 2), 'modified-age'] = df['Age'] + df['Years of Work Experience']
df.loc[(df['Age'] == 23) & (df['Years of Work Experience'] > 3), 'modified-age'] = df['Age'] + df['Years of Work Experience']
df.loc[(df['Age'] == 24) & (df['Years of Work Experience'] > 4), 'modified-age'] = df['Age'] + df['Years of Work Experience']
#......
df = df.loc[df['Annual Salary (Before MBA)']< df['Expected Post-MBA Salary'] ]
# NEW EXPERINCE 
df['new_exp'] = df['Years of Work Experience']
df.loc[(df['Has Management Experience'] == 'Yes') & (df['Years of Work Experience'] == 1) & (df['Current Job Title'] == 'Manager'), 'new_exp'] = df['Age'] - 21
df.loc[(df['Age'] > 24) & (df['Years of Work Experience'] == 0) & (df['Current Job Title'] == 'Manager'), 'new_exp'] = df['Age'] - 21
#n.....
df.loc[(df['modified-age'] < 25) & (df['modified-age'] > 21) & (df['new_exp'] == 0) & (df['Current Job Title'].isin(['Analyst', 'Engineer', 'Consultant'])), 'new_exp'] = 1

df=df.loc[~(df['Gender']=='Other')]


In [4]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df['Has Management Experience'] = le.fit_transform(df['Has Management Experience'])
df['Location Preference (Post-MBA)'] = le.fit_transform(df['Location Preference (Post-MBA)'])
df['Online vs. On-Campus MBA'] = le.fit_transform(df['Online vs. On-Campus MBA'])



In [5]:
ndf = df[['modified-age','Undergraduate Major','Desired Post-MBA Role','Reason for MBA','MBA Funding Source','Current Job Title', 'new_exp','Years of Work Experience','Has Management Experience','Gender','Annual Salary (Before MBA)','Entrepreneurial Interest','Networking Importance','Undergraduate GPA', 'Location Preference (Post-MBA)','Online vs. On-Campus MBA','Decided to Pursue MBA?']]
ndf

,modified-age,Undergraduate Major,Desired Post-MBA Role,Reason for MBA,MBA Funding Source,Current Job Title,new_exp,Years of Work Experience,Has Management Experience,Gender,Annual Salary (Before MBA),Entrepreneurial Interest,Networking Importance,Undergraduate GPA,Location Preference (Post-MBA),Online vs. On-Campus MBA,Decided to Pursue MBA?
0,27,Arts,Finance Manager,Entrepreneurship,Loan,Entrepreneur,8,8,0,1,90624,7.9,7.6,3.18,1,0,Yes
1,24,Arts,Startup Founder,Career Growth,Loan,Analyst,4,4,1,1,53576,3.8,4.1,3.03,1,1,No
2,33,Business,Consultant,Skill Enhancement,Scholarship,Engineer,9,9,0,0,79796,6.7,5.5,3.66,0,1,No
3,31,Engineering,Consultant,Entrepreneurship,Loan,Manager,1,1,0,1,105956,1.0,5.3,2.46,1,0,No
4,28,Business,Consultant,Skill Enhancement,Loan,Entrepreneur,9,9,0,0,96132,9.5,4.9,2.75,0,1,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9993,25,Arts,Startup Founder,Skill Enhancement,Employer,Entrepreneur,8,8,0,1,76025,1.2,7.7,3.40,0,1,Yes
9994,30,Business,Finance Manager,Networking,Self-funded,Entrepreneur,7,7,1,1,92456,1.6,2.7,3.28,1,1,Yes
9996,30,Business,Consultant,Entrepreneurship,Scholarship,Manager,5,5,1,0,82515,7.4,8.5,2.48,0,0,No
9997,31,Economics,Consultant,Networking,Loan,Manager,8,8,1,0,34152,6.8,8.8,2.86,0,0,Yes


In [6]:
x = ndf.iloc[:, :-1].values
y = ndf.iloc[:, -1].values


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1,2,3,4,5])], remainder='passthrough')
x = np.array(ct.fit_transform(x))


In [8]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.25, random_state = 0)

In [9]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)


logistic regression

In [10]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(random_state = 0)
classifier.fit(x_train, y_train)

LogisticRegression(random_state=0)

In [11]:
y_pred = classifier.predict(x_test)

In [12]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[   0  798]
 [   1 1104]]


0.580136626379401

knn classification

In [13]:
from sklearn.neighbors import KNeighborsClassifier
classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
classifier.fit(x_train, y_train)

KNeighborsClassifier()

In [14]:
y_pred = classifier.predict(x_test)

In [15]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[267 531]
 [353 752]]


0.535470310036784

kernal svm 

In [16]:
from sklearn.svm import SVC
classifier = SVC(kernel = 'rbf', random_state = 0)
classifier.fit(x_train, y_train)

SVC(random_state=0)

In [17]:
y_preds = classifier.predict(x_test)

In [18]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_preds)

[[267 531]
 [353 752]]


0.581187598528639

naive bayes

In [19]:
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(x_train, y_train)
y_prednb = classifier.predict(x_test)
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_prednb)

[[267 531]
 [353 752]]


0.5459800315291645

descion tree

In [20]:
from sklearn.tree import DecisionTreeClassifier
classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
classifier.fit(x_train, y_train)
y_predd=classifier.predict(x_test)
cm=confusion_matrix(y_test,y_predd)
print(cm)
accuracy_score(y_test,y_predd)

[[319 479]
 [449 656]]


0.512348922753547

random forest

In [21]:
from sklearn.ensemble import RandomForestClassifier
classifier=RandomForestClassifier(n_estimators=100,criterion='entropy',random_state=0)
classifier.fit(x_train,y_train)
y_rm=classifier.predict(x_test)

In [22]:
print(confusion_matrix(y_test,y_rm))
accuracy_score(y_test,y_rm)

[[113 685]
 [141 964]]


0.5659485023646873